# Baseline: Обучение Multi-Branch MLP на размеченных данных


In [48]:
import os
import sys
import warnings
import pytorch_lightning as pl

warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight

from model import MultiBranchMLP
from data_module import DataModule
from lightning_module import BaseLightningModule

from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping


def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    import random
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
torch.set_float32_matmul_precision('medium')

## 1. Загрузка данных


In [49]:
data_dir = '../data'

dm = DataModule(
    data_dir=data_dir,
    batch_size=128,
    num_workers=0
)

dm.setup()

print(f'Input dimension: {dm.input_dim}')
print(f'Number of classes: {dm.n_classes}')
print(f'Labeled train samples: {len(dm.train_labeled_dataset)}')
print(f'Test samples: {len(dm.test_dataset)}')


Input dimension: 3072
Number of classes: 10
Labeled train samples: 1600
Test samples: 4000
Input dimension: 3072
Number of classes: 10
Labeled train samples: 1600
Test samples: 4000


## 2. Анализ дисбаланса классов и вычисление весов


In [50]:
train_labels = dm.train_labeled_dataset.y

unique_labels = np.unique(train_labels)
class_weights = compute_class_weight(
    'balanced',
    classes=unique_labels,
    y=train_labels
)

print(f'Class weights: {dict(zip(unique_labels, class_weights))}')

class_weights_tensor = torch.FloatTensor(class_weights)
class_weights = class_weights_tensor

Class weights: {np.int64(0): np.float64(1.0738255033557047), np.int64(1): np.float64(0.963855421686747), np.int64(2): np.float64(1.0256410256410255), np.int64(3): np.float64(1.103448275862069), np.int64(4): np.float64(0.9523809523809523), np.int64(5): np.float64(0.935672514619883), np.int64(6): np.float64(0.9523809523809523), np.int64(7): np.float64(1.0596026490066226), np.int64(8): np.float64(0.9248554913294798), np.int64(9): np.float64(1.0457516339869282)}


## 3. Создание модели


## 4. Создание Lightning модуля


In [51]:
checkpoint_callback = ModelCheckpoint(
    dirpath='checkpoints',
    filename='best_model-{epoch:02d}-{val_accuracy:.4f}',
    monitor='val_accuracy',
    mode='max',
    save_top_k=1,
    save_last=True
    )

weight_tensor = torch.tensor(class_weights, dtype=torch.float32)
loss_fn = nn.CrossEntropyLoss(weight=weight_tensor)

def train_model(
    model,
    dm,
    max_epochs=10,
    lr=1e-3,
    optimizer_type='adam'
):

    lightning_model = BaseLightningModule(
        model=model,
        loss_fn=loss_fn,
        optimizer_type=optimizer_type,
        learning_rate=lr,
        task_type='multiclass'
    )



    early_stopping = EarlyStopping(
        monitor='val_f1_macro',
        patience=5,
        mode='max',
    )
    trainer = pl.Trainer(
        precision=16,
        max_epochs=max_epochs,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1,
        callbacks=[early_stopping,checkpoint_callback],
        logger=False,
        enable_progress_bar=False,
        enable_model_summary=False
    )
    trainer.fit(lightning_model, dm)

    metrics = trainer.callback_metrics
    return {
        'val_acc': metrics.get('val_accuracy', 0).item(),
        'val_f1': metrics.get('val_f1_macro', 0).item()
    }


## 5. Обучение модели


In [52]:
hidden_dims = [64,128,256,512]
depths = [2,4,6,8,10,12]
lrs = [1e-2,1e-3,1e-4]
optimizers = ['rmsprop',"adamw"]
combine_modes=['concat','sum']
dropouts = [0.1,0.2,0.3,0.4]
activations = ['relu','gelu']
best_val_f1 = 0.0
best_config = None
max_epochs = 100

hidden_dims = [256]
depths = [4]
lrs = [1e-4]
optimizers = ["adamw"]
combine_modes=['concat']
dropouts = [0.4]
activations = ['gelu']
best_val_f1 = 0.0
best_config = None
max_epochs = 100


for hd in hidden_dims:
    for nb in depths:
        for lr in lrs:
            for opt in optimizers:
                for combine_mode in combine_modes:
                    for dropout in dropouts:
                        for activation in activations:
                             model = MultiBranchMLP(
                                 input_dim=dm.input_dim,
                                 hidden_dim=hd,
                                 output_dim=dm.n_classes,
                                 num_blocks=nb,
                                 dropout=dropout,
                                 combine_mode=combine_mode,
                                 activation=activation
                             )

                             metrics = train_model(
                                 model,
                                 dm,
                                 max_epochs=max_epochs,
                                 lr=lr,
                                 optimizer_type=opt
                             )
                             val_f1 = metrics['val_f1']
                             val_acc = metrics['val_acc']
                             print(f'hidden_dim={hd}, num_blocks={nb}, lr={lr}, opt={opt} -> val_f1={val_f1:.4f}, val_acc={val_acc:.4f} dropout={dropout:.4f} combine_mode={combine_mode} activation={activation}')
                             if val_f1 > best_val_f1:
                                 best_val_f1 = val_f1
                                 best_config = (hd, nb, lr, opt,combine_mode,dropout,activation)

best_hidden_dim, best_depth, best_lr, best_optimizer,best_combine_mode,best_dropout,best_activation = best_config
print(f'Best configuration: hidden_dim={best_hidden_dim}, num_blocks={best_depth}, lr={best_lr}, opt={best_optimizer}, val_f1={best_val_f1:.4f}, best_combine_mode={best_combine_mode}  best_dropout={best_dropout} best_activation={best_activation}')

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Input dimension: 3072
Number of classes: 10
Labeled train samples: 1600
Test samples: 4000
Epoch 0: accuracy=0.1250, f1_macro=0.0485
Epoch 0: accuracy=0.1238, f1_macro=0.0651
Epoch 1: accuracy=0.1509, f1_macro=0.1263
Epoch 2: accuracy=0.1707, f1_macro=0.1485
Epoch 3: accuracy=0.1919, f1_macro=0.1805
Epoch 4: accuracy=0.2040, f1_macro=0.1976
Epoch 5: accuracy=0.2188, f1_macro=0.2118
Epoch 6: accuracy=0.2239, f1_macro=0.2207
Epoch 7: accuracy=0.2301, f1_macro=0.2273
Epoch 8: accuracy=0.2314, f1_macro=0.2305
Epoch 9: accuracy=0.2356, f1_macro=0.2334
Epoch 10: accuracy=0.2395, f1_macro=0.2389
Epoch 11: accuracy=0.2415, f1_macro=0.2413
Epoch 12: accuracy=0.2448, f1_macro=0.2450
Epoch 13: accuracy=0.2486, f1_macro=0.2487
Epoch 14: accuracy=0.2507, f1_macro=0.2505
Epoch 15: accuracy=0.2552, f1_macro=0.2553
Epoch 16: accuracy=0.2569, f1_macro=0.2569
Epoch 17: accuracy=0.2595, f1_macro=0.2595
Epoch 18: accuracy=0.2620, f1_macro=0.2619
Epoch 19: accuracy=0.2636, f1_macro=0.2639
Epoch 20: accurac

`Trainer.fit` stopped: `max_epochs=100` reached.


hidden_dim=256, num_blocks=4, lr=0.0001, opt=adamw -> val_f1=0.3158, val_acc=0.3150 dropout=0.4000 combine_mode=concat activation=gelu
Best configuration: hidden_dim=256, num_blocks=4, lr=0.0001, opt=adamw, val_f1=0.3158, best_combine_mode=concat  best_dropout=0.4 best_activation=gelu


## 6. Оценка на тестовой выборке


In [53]:
best_model_path = checkpoint_callback.best_model_path
print(f'Loading best model from: {best_model_path}')

model = MultiBranchMLP(
    input_dim=dm.input_dim,
    hidden_dim=best_hidden_dim,
    output_dim=dm.n_classes,
    num_blocks=best_depth,
    dropout=best_dropout,
    combine_mode=best_combine_mode,
    activation=best_activation
)

best_model = BaseLightningModule.load_from_checkpoint(
    best_model_path,
    model=model,
    loss_fn=loss_fn,
    optimizer_type='adamw',
    learning_rate=1e-3,
    task_type='multiclass'
)
early_stopping = EarlyStopping(
    monitor='val_f1_macro',
    patience=5,
    mode='max',
)

trainer = Trainer(
    max_epochs=max_epochs,
    callbacks=[checkpoint_callback,early_stopping],
    enable_checkpointing=True,
    logger=True,
    enable_progress_bar=True,
    enable_model_summary=True,
    accelerator='auto',
    devices='auto'
)

test_results = trainer.test(best_model, dm)

print('\n=== Финальные результаты на тестовой выборке ===')
for key, value in test_results[0].items():
    print(f'{key}: {value:.4f}')


Loading best model from: E:\PythonProjects\DeepMachineLearningSSD\lesson7\homework\baseline\checkpoints\best_model-epoch=99-val_accuracy=0.3150.ckpt


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Input dimension: 3072
Number of classes: 10
Labeled train samples: 1600
Test samples: 4000


Testing: |          | 0/? [00:00<?, ?it/s]

Test results: accuracy=0.3275, f1_macro=0.3302


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       test_accuracy       │    0.32749998569488525    │
│       test_f1_macro       │    0.3302322030067444     │
│         test_loss         │    10.118412971496582     │
└───────────────────────────┴───────────────────────────┘


=== Финальные результаты на тестовой выборке ===
test_loss: 10.1184
test_accuracy: 0.3275
test_f1_macro: 0.3302
